In [8]:
import os
import glob
import numpy as np
import pandas as pd

def _nanmean(arr: np.ndarray) -> float:
    return float(np.nanmean(arr)) if arr.size > 0 else float("nan")

def summarize_dataset(npz_path: str) -> dict:
    """
    Load one dataset .npz and compute summary statistics.
    Returns a dict suitable for creating a pandas DataFrame row.
    """
    dsname = os.path.splitext(os.path.basename(npz_path))[0]
    data = np.load(npz_path)

    edu_avg = np.array(data["edu_avg_grid"], dtype=float)
    f1      = np.array(data["f1_all_grid"], dtype=float)
    edu_r   = np.array(data["edu_avg_grid_r"], dtype=float)
    f1_r    = np.array(data["f1_all_grid_r"], dtype=float)

    # Align shapes / masks (ignore NaNs)
    mask = ~(np.isnan(edu_avg) | np.isnan(edu_r) | np.isnan(f1) | np.isnan(f1_r))
    if not np.any(mask):
        n = 0
        f1_mean, f1_r_mean, f1_gain_mean = np.nan, np.nan, np.nan
        edu_mean, edu_r_mean, edu_drop_mean = np.nan, np.nan, np.nan
    else:
        f1_vals, f1r_vals = f1[mask], f1_r[mask]
        edu_vals, edur_vals = edu_avg[mask], edu_r[mask]

        f1_mean      = float(np.mean(f1_vals))
        f1_r_mean    = float(np.mean(f1r_vals))
        f1_gain_mean = (float(np.mean(f1r_vals - f1_vals)))/f1_mean

        edu_mean      = float(np.mean(edu_vals))
        edu_r_mean    = float(np.mean(edur_vals))
        edu_drop_mean = (float(np.mean(edu_vals - edur_vals)))/edu_mean
        n = int(mask.sum())

    return {
        "dataset": dsname,
        "n_configs": n,
        "f1_base_mean": f1_mean,
        "f1_r_mean": f1_r_mean,
        "f1_gain_mean": f1_gain_mean,
        "edu_base_mean": edu_mean,
        "edu_r_mean": edu_r_mean,
        "edu_drop_mean": edu_drop_mean,
    }

def compile_summary_dataframe(base_dir: str, dnsname: str) -> pd.DataFrame:
    """
    Scan all saved datasets under {base_dir}/{dnsname} and build a summary DataFrame.
    Adds a macro average row (unweighted mean across datasets).
    """
    folder = os.path.join(base_dir, dnsname)
    npz_files = sorted(glob.glob(os.path.join(folder, "*.npz")))

    if not npz_files:
        raise FileNotFoundError(f"No saved datasets found in: {folder}")

    rows = [summarize_dataset(p) for p in npz_files]
    df = pd.DataFrame(rows)

    # Macro average row (unweighted mean across datasets)
    macro = {
        "dataset": "Macro avg",
        #"n_configs": int(df["n_configs"].sum()),  # total configs across datasets
        "f1_base_mean": df["f1_base_mean"].mean(),
        "f1_r_mean": df["f1_r_mean"].mean(),
        "f1_gain_mean": df["f1_gain_mean"].mean(),
        "edu_base_mean": df["edu_base_mean"].mean(),
        "edu_r_mean": df["edu_r_mean"].mean(),
        "edu_drop_mean": df["edu_drop_mean"].mean(),
    }
    df = pd.concat([df, pd.DataFrame([macro])], ignore_index=True)
    return df

def dataframe_to_latex(df: pd.DataFrame,
                       out_path: str,
                       caption: str = "Average F1 gain and CI-EDU drop per dataset.",
                       label: str = "tab:f1_edu_summary",
                       float_fmt="{:.4f}") -> str:
    """
    Save LaTeX table to out_path and return the LaTeX string.
    """
    cols = [
        "dataset", #"n_configs",
        "f1_base_mean", "f1_r_mean", "f1_gain_mean",
        "edu_base_mean", "edu_r_mean", "edu_drop_mean",
    ]
    # Format floats
    df_fmt = df.copy()
    for c in cols:
        if df_fmt[c].dtype.kind in "f":
            df_fmt[c] = df_fmt[c].map(lambda x: float_fmt.format(x) if pd.notnull(x) else "")

    latex = df_fmt[cols].to_latex(index=False,
                                  caption=caption,
                                  label=label,
                                  escape=True)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(latex)
    return latex
    

In [9]:
# base_dir and dnsname must match what you used for saving
dnsname = "grid-data"   # folder name grouping a set of datasets
base_dir = "./results"  # local or Google Drive


df = compile_summary_dataframe(base_dir, dnsname)
latex_path = os.path.join(base_dir, dnsname, "summary_table.tex")
latex_str = dataframe_to_latex(
    df,
    out_path=latex_path,
    caption="Average F1 gain ($\\overline{\\Delta \\mathrm{F1}}$) and CI-EDU drop ($\\overline{\\Delta \\mathrm{EDU}}$) per dataset.",
    label="tab:f1_gain_edu_drop"
)
print(df)
print("\nLaTeX saved to:", latex_path)

           dataset  n_configs  f1_base_mean  f1_r_mean  f1_gain_mean  \
0    breast_cancer      300.0      0.860429   0.848129     -0.014295   
1             iris      120.0      0.931852   0.931852      0.000000   
2  openml_credit_g      300.0      0.835111   0.836767      0.001982   
3     openml_sonar      180.0      0.735450   0.737125      0.002278   
4             pima      300.0      0.831746   0.828730     -0.003626   
5             wine      150.0      0.774074   0.772469     -0.002073   
6        Macro avg        NaN      0.828110   0.825845     -0.002622   

   edu_base_mean  edu_r_mean  edu_drop_mean  
0       0.007987    0.006085       0.238110  
1       0.018499    0.016255       0.121261  
2       0.038815    0.031006       0.201171  
3       0.081739    0.075571       0.075458  
4       0.040924    0.033260       0.187272  
5       0.018478    0.015503       0.161009  
6       0.034407    0.029613       0.164047  

LaTeX saved to: ./results/grid-data/summary_table.tex


In [10]:
latex_str

'\\begin{table}\n\\caption{Average F1 gain ($\\overline{\\Delta \\mathrm{F1}}$) and CI-EDU drop ($\\overline{\\Delta \\mathrm{EDU}}$) per dataset.}\n\\label{tab:f1_gain_edu_drop}\n\\begin{tabular}{lllllll}\n\\toprule\ndataset & f1\\_base\\_mean & f1\\_r\\_mean & f1\\_gain\\_mean & edu\\_base\\_mean & edu\\_r\\_mean & edu\\_drop\\_mean \\\\\n\\midrule\nbreast\\_cancer & 0.8604 & 0.8481 & -0.0143 & 0.0080 & 0.0061 & 0.2381 \\\\\niris & 0.9319 & 0.9319 & 0.0000 & 0.0185 & 0.0163 & 0.1213 \\\\\nopenml\\_credit\\_g & 0.8351 & 0.8368 & 0.0020 & 0.0388 & 0.0310 & 0.2012 \\\\\nopenml\\_sonar & 0.7354 & 0.7371 & 0.0023 & 0.0817 & 0.0756 & 0.0755 \\\\\npima & 0.8317 & 0.8287 & -0.0036 & 0.0409 & 0.0333 & 0.1873 \\\\\nwine & 0.7741 & 0.7725 & -0.0021 & 0.0185 & 0.0155 & 0.1610 \\\\\nMacro avg & 0.8281 & 0.8258 & -0.0026 & 0.0344 & 0.0296 & 0.1640 \\\\\n\\bottomrule\n\\end{tabular}\n\\end{table}\n'